# PhysAR-IDS -- NB1b (v6): Split-sensitivity study -- how much does the evaluation protocol move the F1?

**Question.** The headline v6 numbers (XGBoost 0.849, RF 0.861) come from one protocol: the last 15% of every `(class, PLC)` series is the test set, with a 1 s embargo.
This notebook re-evaluates the **same 93 features and the same models** under 27 different splits, so the dissertation can say how much of the score depends on the protocol,
and quantify how much a careless random split would inflate it (the reviewer's leakage / session-artefact concern).

**Inputs:** `phase1_data_v6.pkl` (NB1 v6). Train and test are re-assembled into one table (the embargo dropped 0 rows, so nothing is missing). Tree models are invariant to per-feature affine scaling, so the stored scaled values can be reused for every split without refitting a scaler.

## Splits compared

| Family | Splits | What it tests |
|---|---|---|
| **Tail fraction** | test = last 10 / **15 (reference)** / 20 / 25 / 30 / 40% of each series, 1 s embargo; the **80/20, 70/30 and 60/40** train/test ratios are the 20, 30 and 40% rows | How much the score depends on how much of the future is held out (more test = training stops earlier). |
| **Embargo** | 0 / 1 / 5 / 10 / 30 s at 15% | Physical samples arrive in bursts ~5 s apart, so 0-1 s embargoes drop nothing; 5 s and more finally remove neighbouring bursts. Shows whether adjacency leaks. |
| **Direction** | test = **first** 15%; test = **middle** 15% (train on both sides) | Whether the score depends on the test block being the *end* of the recording (drift) or is the same wherever you cut. |
| **Rolling origin** | train < 55% -> test 55-70%; < 70% -> 70-85%; < 85% -> 85-100% | Performance on several consecutive future blocks, not one. |
| **Random (leaky by design)** | random rows and random bursts at 15, 20, 30 and 40% test (so 80/20, 70/30, 60/40 are also shown the "wrong" way); random 50 s blocks at 15% | The "wrong" protocols. Random rows put copies of the same ~229-row burst on both sides. Quantifies the inflation. |
| **Leave-PLC-out** | train on 6 PLCs, test on the 3 held-out PLCs (3 folds), all times | Generalisation to unseen devices -- a harder, different question. |

## Models and feature sets
XGBoost (NB1 baseline settings, CPU `hist`) on **all 93 / physical-only (33) / cyber-only (60)** features, and RandomForest (NB1 baseline settings) on all 93. NB1 baseline settings are used on purpose:
the tuned parameters from NB2 were chosen on the reference split, so applying them to other splits would favour that split. 5 seeds each; for random splits the seed also draws the split.

## Diagnostics reported with every split (before any model is trained)
* rows / bursts in train and test; test class counts;
* **burst overlap** = % of test rows whose `(class, PLC, timestamp)` burst also appears in train (direct measure of duplicate leakage);
* **median / min time from a test burst to the nearest train burst** of the same series (seconds).

## Caveats to carry into the dissertation
* The deterministic splits have only model-seed variance; the dominant uncertainty is the **choice of split**, shown by the spread across rows of the table, not by the +/- column.
* Each split has a different test set, so differences between splits are descriptive, not paired tests.
* Only ~2,700 distinct bursts exist; an F1 difference of a few points between two splits can be a handful of bursts.

## 0. Setup

In [ ]:
import os, sys, time, json, platform, warnings
import numpy as np
import pandas as pd
import joblib
from scipy import stats
warnings.filterwarnings('ignore')
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import xgboost as xgb
import matplotlib.pyplot as plt

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = False           # True = tiny plumbing run; never use for results
if SMOKE_TEST: SEEDS = SEEDS[:2]
print(f'xgboost {xgb.__version__} | cwd {os.getcwd()} | seeds {SEEDS} | SMOKE_TEST={SMOKE_TEST}')

## 1. Load NB1 v6 and rebuild the single table (train + test)

In [ ]:
d = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert d.get('PIPELINE_VERSION') == PIPELINE_VERSION and d.get('TEST_FRAC') == 0.15, (d.get('PIPELINE_VERSION'), d.get('TEST_FRAC'))
FEATURE_COLS = d['FEATURE_COLS']
X = np.vstack([d['X_train_scaled'], d['X_test_scaled']]).astype(np.float32)
y = np.concatenate([d['y_train'], d['y_test']]).astype(int)
plc = np.concatenate([d['PLCID_train'], d['PLCID_test']]).astype(int)
ts = np.concatenate([d['timestamp_train'], d['timestamp_test']]).astype(float)
orig_is_test = np.concatenate([np.zeros(len(d['y_train']), bool), np.ones(len(d['y_test']), bool)])
if SMOKE_TEST:
    k = np.arange(0, len(X), 6); X, y, plc, ts, orig_is_test = X[k], y[k], plc[k], ts[k], orig_is_test[k]
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
CYBER_IDX = [i for i, c in enumerate(FEATURE_COLS) if c.startswith('cyber_')]
PHYS_IDX = [i for i in range(len(FEATURE_COLS)) if i not in set(CYBER_IDX)]
FEATSETS = {'all': list(range(len(FEATURE_COLS))), 'physical_only': PHYS_IDX, 'cyber_only': CYBER_IDX}
X_fs = {k: np.ascontiguousarray(X[:, v]) for k, v in FEATSETS.items()}
print(f'rows {len(X)} | features all/physical/cyber = {len(FEATURE_COLS)}/{len(PHYS_IDX)}/{len(CYBER_IDX)}')

# per-row series info
g = pd.DataFrame({'y': y, 'p': plc, 't': ts})
gg = g.groupby(['y', 'p'])['t']
tmin_g = gg.transform('min').values; tmax_g = gg.transform('max').values
u = (ts - tmin_g) / (tmax_g - tmin_g + 1e-12)                  # position in its series, 0..1 (over ALL rows)
burst_id = pd.factorize(pd.MultiIndex.from_arrays([y, plc, ts]))[0]
print(f'series (class x PLC): {g.groupby(["y", "p"]).ngroups} | distinct bursts: {burst_id.max() + 1}')

## 2. Split definitions

All temporal splits are computed per `(class, PLC)` series from the series' own time range, exactly as in NB1 (`cutoff = t_min + (t_max - t_min) * (1 - test_frac)`; rows within `+/- embargo` seconds of the cut are dropped from both sides).
The reference split is checked against the one NB1 actually used.

In [ ]:
def tail_split(f, e):
    cut = tmin_g + (tmax_g - tmin_g) * (1 - f)
    emb = (ts >= cut - e) & (ts < cut + e); te = (ts >= cut) & ~emb; tr = (ts < cut) & ~emb
    return tr, te

def head_split(f, e):
    cut = tmin_g + (tmax_g - tmin_g) * f
    emb = (ts >= cut - e) & (ts < cut + e); te = (ts < cut) & ~emb; tr = (ts >= cut) & ~emb
    return tr, te

def middle_split(f, e, centre=0.5):
    a, b = centre - f / 2, centre + f / 2
    ta, tb = tmin_g + (tmax_g - tmin_g) * a, tmin_g + (tmax_g - tmin_g) * b
    emb = ((ts >= ta - e) & (ts < ta + e)) | ((ts >= tb - e) & (ts < tb + e))
    te = (ts >= ta) & (ts < tb) & ~emb; tr = ((ts < ta) | (ts >= tb)) & ~emb
    return tr, te

def rolling_split(b, c, dd, e=1.0):
    cut = tmin_g + (tmax_g - tmin_g) * b
    tr = ts < cut - e; te = (u >= c) & (u < dd) & (ts >= cut + e)
    return tr, te

def random_row_split(f, seed):
    idx = np.arange(len(y)); _, te_i = train_test_split(idx, test_size=f, stratify=y, random_state=seed)
    te = np.zeros(len(y), bool); te[te_i] = True; return ~te, te

def random_group_split(group_ids, f, seed):
    """choose f of the groups per class at random as test"""
    rng = np.random.RandomState(seed); te = np.zeros(len(y), bool)
    for c in np.unique(y):
        gs = np.unique(group_ids[y == c]); pick = rng.choice(gs, size=int(round(f * len(gs))), replace=False)
        te |= (y == c) & np.isin(group_ids, pick)
    return ~te, te

block50 = pd.factorize(pd.MultiIndex.from_arrays([y, plc, ((ts - tmin_g) // 50.0).astype(int)]))[0]

def plc_holdout(held):
    te = np.isin(plc, held); return ~te, te

SPLITS = []        # (name, family, label, fn(seed)->(train_mask, test_mask))
def add(name, fam, fn): SPLITS.append((name, fam, fn))
def ratio_tag(p): return f' [{100 - p}/{p} train/test]' if p in (20, 30, 40) else ''
for f in [0.10, 0.15, 0.20, 0.25, 0.30, 0.40]:
    p = int(round(f * 100)); add(f'tail f={p}% emb=1s{ratio_tag(p)}', 'tail fraction', (lambda f: lambda s: tail_split(f, 1.0))(f))
for e in [0, 5, 10, 30]:                  add(f'tail f=15% emb={e}s', 'embargo', (lambda e: lambda s: tail_split(0.15, float(e)))(e))
add('head f=15% emb=1s', 'direction', lambda s: head_split(0.15, 1.0))
add('middle f=15% emb=1s', 'direction', lambda s: middle_split(0.15, 1.0))
for (b, c, dd) in [(0.55, 0.55, 0.70), (0.70, 0.70, 0.85), (0.85, 0.85, 1.0001)]:
    add(f'rolling train<{int(b * 100)}% test {int(c * 100)}-{min(int(dd * 100), 100)}%', 'rolling origin', (lambda b, c, dd: lambda s: rolling_split(b, c, dd))(b, c, dd))
add('RANDOM rows 15%', 'random (leaky)', lambda s: random_row_split(0.15, s))
add('RANDOM bursts 15%', 'random (leaky)', lambda s: random_group_split(burst_id, 0.15, s))
add('RANDOM 50 s blocks 15%', 'random (leaky)', lambda s: random_group_split(block50, 0.15, s))
for f in [0.20, 0.30, 0.40]:
    p = int(round(f * 100))
    add(f'RANDOM rows {p}%{ratio_tag(p)}', 'random (leaky)', (lambda f: lambda s: random_row_split(f, s))(f))
    add(f'RANDOM bursts {p}%{ratio_tag(p)}', 'random (leaky)', (lambda f: lambda s: random_group_split(burst_id, f, s))(f))
for held in [(1, 2, 3), (4, 5, 6), (7, 8, 9)]: add(f'leave-PLC-out test PLCs {held}', 'leave-PLC-out', (lambda h: lambda s: plc_holdout(h))(held))
REF = 'tail f=15% emb=1s'
print(len(SPLITS), 'splits')

tr0, te0 = dict((n, fn) for n, _, fn in SPLITS)[REF](0)
agree = (te0 == orig_is_test).mean()
if SMOKE_TEST: print(f'(smoke test: thinned rows shift the cut points; agreement with NB1 split {agree:.4f})')
else:
    assert np.array_equal(te0, orig_is_test) and np.array_equal(tr0, ~orig_is_test), 'reference split does not reproduce NB1 split'
    print('Reference split reproduces the NB1 v6 split exactly.')

## 3. Split diagnostics (no model trained yet)

For each split (seed 42 for the random ones): sizes, test class counts, **burst overlap** (% of test rows whose burst also appears in train) and the **time from each test burst to the nearest train burst** of the same series.

In [ ]:
def nearest_gap(tr, te):
    """seconds from each distinct test burst to the nearest train timestamp of the same (class, PLC); NaN if the series has no train rows"""
    out = []
    df = pd.DataFrame({'y': y[te], 'p': plc[te], 't': ts[te]}).drop_duplicates()
    trdf = pd.DataFrame({'y': y[tr], 'p': plc[tr], 't': ts[tr]}).drop_duplicates()
    trg = {k: np.sort(v['t'].values) for k, v in trdf.groupby(['y', 'p'])}
    for k, v in df.groupby(['y', 'p']):
        a = trg.get(k)
        if a is None or len(a) == 0: continue
        t = v['t'].values; i = np.searchsorted(a, t)
        lo = np.where(i > 0, np.abs(t - a[np.clip(i - 1, 0, len(a) - 1)]), np.inf); hi = np.where(i < len(a), np.abs(a[np.clip(i, 0, len(a) - 1)] - t), np.inf)
        out.append(np.minimum(lo, hi))
    return np.concatenate(out) if out else np.array([np.nan])

diag_rows = []
for name, fam, fn in SPLITS:
    tr, te = fn(SEEDS[0]); gap = nearest_gap(tr, te)
    diag_rows.append({'split': name, 'family': fam, 'train_rows': int(tr.sum()), 'test_rows': int(te.sum()), 'dropped_rows': int(len(y) - tr.sum() - te.sum()),
                      'test_bursts': int(len(np.unique(burst_id[te]))), 'train_bursts': int(len(np.unique(burst_id[tr]))),
                      'burst_overlap_%': round(100 * np.isin(burst_id[te], np.unique(burst_id[tr])).mean(), 2),
                      'median_gap_s': round(float(np.nanmedian(gap)), 2), 'min_gap_s': round(float(np.nanmin(gap)), 2),
                      'test_classes': np.bincount(y[te], minlength=6).tolist()})
diag = pd.DataFrame(diag_rows); pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 60)
print(diag.drop(columns=['test_classes']).to_string(index=False))
print('\nTest class counts (rows):'); print(diag[['split', 'test_classes']].to_string(index=False))
diag.to_csv(f'split_diagnostics_{PIPELINE_VERSION}.csv', index=False)

## 4. Train and evaluate: 4 model/feature-set combinations x 27 splits x 5 seeds

In [ ]:
def make_xgb(seed):
    n_est = 30 if SMOKE_TEST else 200
    return xgb.XGBClassifier(n_estimators=n_est, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, objective='multi:softprob',
                             num_class=6, random_state=seed, n_jobs=-1, tree_method='hist')
def make_rf(seed): return RandomForestClassifier(n_estimators=(30 if SMOKE_TEST else 200), max_depth=12, random_state=seed, n_jobs=-1)
MODELS = [('XGBoost', 'all', make_xgb), ('XGBoost', 'physical_only', make_xgb), ('XGBoost', 'cyber_only', make_xgb), ('RandomForest', 'all', make_rf)]

def metrics(yt, yp):
    b = (yt == 0); a = ~b
    return {'f1': f1_score(yt, yp, average='weighted'), 'macro_f1': f1_score(yt, yp, average='macro'),
            'benign_fpr': float((yp[b] != 0).mean()) if b.any() else np.nan, 'attack_miss': float((yp[a] == 0).mean()) if a.any() else np.nan,
            'per_class_f1': f1_score(yt, yp, average=None, labels=range(6), zero_division=0)}

RES = {}     # RES[split][(model, fs)][seed] = metrics
t_all = time.time()
for si, (name, fam, fn) in enumerate(SPLITS):
    RES[name] = {}
    for seed in SEEDS:
        tr, te = fn(seed)
        for mname, fs, mk in MODELS:
            t0 = time.time(); Xa = X_fs[fs]
            m = mk(seed).fit(Xa[tr], y[tr]); r = metrics(y[te], m.predict(Xa[te]))
            RES[name].setdefault((mname, fs), {})[seed] = r
    line = '  '.join(f'{mname[:3]}-{fs[:4]} {np.mean([RES[name][(mname, fs)][s]["f1"] for s in SEEDS]):.4f}' for mname, fs, _ in MODELS)
    print(f'[{si + 1:>2}/{len(SPLITS)}] {name:<38} {line}   ({(time.time() - t_all) / 60:.1f} min elapsed)')
    joblib.dump({'SEEDS': SEEDS, 'RES': RES}, f'split_sensitivity_partial_{PIPELINE_VERSION}.pkl')       # checkpoint

## 5. Results

In [ ]:
T4 = stats.t.ppf(0.975, df=len(SEEDS) - 1)
def f1s(split, key, metric='f1'): return np.array([RES[split][key][s][metric] for s in SEEDS])
key_cols = [(m, fs) for m, fs, _ in MODELS]
rows = []
for name, fam, _ in SPLITS:
    r = {'split': name, 'family': fam}
    for k in key_cols:
        a = f1s(name, k); r[f'{k[0]}|{k[1]}'] = f'{a.mean():.4f} +/- {a.std(ddof=1):.4f}'
        r[f'{k[0]}|{k[1]}_mean'] = a.mean()
    rows.append(r)
summ = pd.DataFrame(rows)
show = ['split', 'family'] + [f'{m}|{fs}' for m, fs in key_cols]
print(summ[show].to_string(index=False))
full_tab = summ.merge(diag.drop(columns=['test_classes', 'family']), on='split'); full_tab.to_csv(f'split_sensitivity_summary_{PIPELINE_VERSION}.csv', index=False)

print('\nFamily means of the XGBoost (all 93) F1:')
fam = summ.groupby('family', sort=False)['XGBoost|all_mean'].agg(['mean', 'min', 'max', 'count']).round(4); print(fam.to_string())

In [ ]:
# benign false-alarm and attack-miss rates, XGBoost all features
rows = []
for name, fam, _ in SPLITS:
    rows.append({'split': name, 'benign FPR': f'{f1s(name, ("XGBoost", "all"), "benign_fpr").mean():.4f}', 'attack miss': f'{f1s(name, ("XGBoost", "all"), "attack_miss").mean():.4f}',
                 'macro F1': f'{f1s(name, ("XGBoost", "all"), "macro_f1").mean():.4f}'})
print(pd.DataFrame(rows).to_string(index=False))

print('\nPer-class F1, XGBoost (all 93), mean over seeds:')
pc = pd.DataFrame({n: np.stack([RES[n][('XGBoost', 'all')][s]['per_class_f1'] for s in SEEDS]).mean(0) for n, _, _ in SPLITS}, index=class_names).T
print(pc.round(3).to_string()); pc.to_csv(f'split_sensitivity_per_class_{PIPELINE_VERSION}.csv')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9)); names = [n for n, _, _ in SPLITS]; ypos = np.arange(len(names)); h = 0.2
colors = {('XGBoost', 'all'): '#1f77b4', ('RandomForest', 'all'): '#2ca02c', ('XGBoost', 'physical_only'): '#ff7f0e', ('XGBoost', 'cyber_only'): '#9467bd'}
for j, k in enumerate(key_cols):
    mu = [f1s(n, k).mean() for n in names]; sd = [f1s(n, k).std(ddof=1) for n in names]
    ax.barh(ypos + (j - 1.5) * h, mu, h, xerr=sd, color=colors[k], label=f'{k[0]} ({k[1]})', capsize=1.5)
ax.set_yticks(ypos); ax.set_yticklabels(names, fontsize=8); ax.invert_yaxis(); ax.set_xlim(0, 1.02); ax.set_xlabel('weighted F1 on the held-out part (mean +/- std over seeds)')
ref_i = names.index(REF); ax.axhline(ref_i, color='gray', lw=0.6, ls=':')
ax.axvline(f1s(REF, ('XGBoost', 'all')).mean(), color='#1f77b4', ls='--', lw=0.8); ax.legend(loc='lower right', fontsize=8); ax.set_title('Split sensitivity (v6, 93 features)')
plt.tight_layout(); plt.savefig(f'split_sensitivity_{PIPELINE_VERSION}.png', dpi=150); plt.show()

## 6. What the numbers say (generated from the results above, no hand-written conclusions)

In [ ]:
x = lambda n: f1s(n, ('XGBoost', 'all')).mean(); xp = lambda n: f1s(n, ('XGBoost', 'physical_only')).mean(); xc = lambda n: f1s(n, ('XGBoost', 'cyber_only')).mean(); rf = lambda n: f1s(n, ('RandomForest', 'all')).mean()
ref = x(REF); dg = diag.set_index('split')
print(f'Reference ({REF}): XGBoost {ref:.4f} | RF {rf(REF):.4f} | physical-only {xp(REF):.4f} | cyber-only {xc(REF):.4f}\n')
def block(title, names):
    print(title)
    for n in names: print(f'   {n:<38} XGB {x(n):.4f} ({x(n) - ref:+.4f})  RF {rf(n):.4f}  phys-only {xp(n):.4f}  cyber-only {xc(n):.4f}   burst-overlap {dg.loc[n, "burst_overlap_%"]:.1f}%  median gap {dg.loc[n, "median_gap_s"]}s')
block('Tail fraction:', [n for n, f, _ in SPLITS if f == 'tail fraction'])
block('Embargo:', [n for n, f, _ in SPLITS if f == 'embargo'])
block('Direction:', [n for n, f, _ in SPLITS if f == 'direction'])
block('Rolling origin:', [n for n, f, _ in SPLITS if f == 'rolling origin'])
block('Random splits (leaky by design):', [n for n, f, _ in SPLITS if f == 'random (leaky)'])
block('Leave-PLC-out:', [n for n, f, _ in SPLITS if f == 'leave-PLC-out'])

print('\nThe classic ratios, temporal vs random (same test fraction):')
for p in (20, 30, 40):
    t_ = [n for n, f, _ in SPLITS if n.startswith(f'tail f={p}%')][0]; r_ = f'RANDOM rows {p}% [{100 - p}/{p} train/test]'; b_ = f'RANDOM bursts {p}% [{100 - p}/{p} train/test]'
    print(f'   {100 - p}/{p}:  temporal {x(t_):.4f} (RF {rf(t_):.4f}, phys-only {xp(t_):.4f}, cyber-only {xc(t_):.4f}) | random rows {x(r_):.4f} (phys-only {xp(r_):.4f}) | random bursts {x(b_):.4f}')
tails = [x(n) for n, f, _ in SPLITS if f == 'tail fraction']
print(f'\nRange of XGBoost F1 over the tail-fraction sweep: {min(tails):.4f} - {max(tails):.4f} (spread {max(tails) - min(tails):.4f}); seed std at the reference is {f1s(REF, ("XGBoost", "all")).std(ddof=1):.4f}.')
emb = [x(n) for n, f, _ in SPLITS if f == 'embargo']
print(f'Range over the embargo sweep: {min(emb):.4f} - {max(emb):.4f} (spread {max(emb) - min(emb):.4f}).')
rr = x('RANDOM rows 15%'); print(f'Random-row split inflates XGBoost by {rr - ref:+.4f} over the reference; physical-only by {xp("RANDOM rows 15%") - xp(REF):+.4f}, cyber-only by {xc("RANDOM rows 15%") - xc(REF):+.4f}.')
pl = np.mean([x(n) for n, f, _ in SPLITS if f == 'leave-PLC-out']); print(f'Mean leave-PLC-out XGBoost F1: {pl:.4f} ({pl - ref:+.4f} vs reference).')
ro = [x(n) for n, f, _ in SPLITS if f == 'rolling origin']; print(f'Rolling-origin blocks: {[round(float(v), 4) for v in ro]} (mean {np.mean(ro):.4f}, spread {max(ro) - min(ro):.4f}).')
print('\nReport the reference split as the primary result, the rolling-origin and leave-PLC-out rows as robustness, and the random-split rows only as a demonstration of leakage.')

## 7. Save

In [ ]:
joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'REF': REF, 'splits': [(n, f) for n, f, _ in SPLITS], 'RES': RES, 'diagnostics': diag, 'summary': full_tab},
            f'split_sensitivity_results_{PIPELINE_VERSION}.pkl')
json.dump({'notebook': 'NB1b_split_sensitivity', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'n_splits': len(SPLITS), 'xgboost': xgb.__version__,
           'python': platform.python_version(), 'smoke_test': SMOKE_TEST, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())},
          open(f'split_sensitivity_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'split_sensitivity_results_{PIPELINE_VERSION}.pkl', f'split_sensitivity_summary_{PIPELINE_VERSION}.csv', f'split_diagnostics_{PIPELINE_VERSION}.csv',
          f'split_sensitivity_per_class_{PIPELINE_VERSION}.csv', f'split_sensitivity_{PIPELINE_VERSION}.png', f'split_sensitivity_manifest_{PIPELINE_VERSION}.json']:
    print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)